# QDM Phase 2A + 2A.5: Pythia-70M Streaming Bit-Width Sweep

This notebook is the **memory-safe revised version**. It does **not** store full `tokens × SAE features` tensors. Instead, it computes per-feature Pearson correlations with streaming/running sums.

Outputs:
- `phase2a_L4_summary.csv`
- `phase2a_L4_per_feature.csv`
- `phase2a5_L2_summary.csv`
- `phase2a5_L2_per_feature.csv`
- layer comparison plots
- combined results table

## 1. Install dependencies

In [ ]:
!pip install -q transformer_lens sae-lens datasets matplotlib pandas tqdm scikit-learn

## 2. Imports and device check

In [ ]:
import os
import gc
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformer_lens import HookedTransformer
from sae_lens import SAE
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "No GPU available. On Vast/Colab, select a GPU runtime."
device = "cuda"
torch.set_grad_enabled(False)

print("Device:", torch.cuda.get_device_name(0))
try:
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
except Exception:
    pass

## 3. Config

In [ ]:
MODEL_NAME = "pythia-70m-deduped"
SAE_RELEASE = "pythia-70m-deduped-res-sm"

# Pythia-70M has layers 0-5. Layer 4 was the smoke-test layer; layer 2 is the secondary robustness check.
PRIMARY_LAYER = 4
SECONDARY_LAYER = 2

# Phase 2A proper-scale run. If debugging, temporarily set to 50_000.
TOKEN_BUDGET = 200_000
SEQ_LEN = 512

# Streaming batch size. If the kernel dies, reduce to 1. If it is stable and slow, try 4.
BATCH_SIZE = 2

BITS_TO_TEST = [8, 7, 6, 5]
FIRING_THRESHOLD = 0.001

# Results directory. Works on Vast (/workspace), Colab, or local Jupyter.
if os.path.exists("/workspace"):
    RESULTS_DIR = "/workspace/qdm_phase2a_results"
else:
    RESULTS_DIR = "./qdm_phase2a_results"

os.makedirs(RESULTS_DIR, exist_ok=True)
CKPT = lambda name: os.path.join(RESULTS_DIR, name)

print(f"Model: {MODEL_NAME}")
print(f"SAE release: {SAE_RELEASE}")
print(f"Layers: primary={PRIMARY_LAYER}, secondary={SECONDARY_LAYER}")
print(f"Tokens: {TOKEN_BUDGET:,}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Bit-widths: FP16 + {BITS_TO_TEST}")
print(f"Results dir: {RESULTS_DIR}")

## 4. Load model and tokens

In [ ]:
print("Loading model...")
model = HookedTransformer.from_pretrained(MODEL_NAME, device=device)
model.eval()
print(f"Model loaded: n_layers={model.cfg.n_layers}, d_model={model.cfg.d_model}")

# Save original weights once. For Pythia-70M this is small enough.
original_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
print("Original state saved.")

print("
Loading and tokenizing WikiText-2 train split...")
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
full_text = "

".join(x for x in ds["text"] if len(x.strip()) > 0)

# Use tokenizer directly; model.to_tokens truncates to context length.
token_ids = model.tokenizer.encode(full_text, add_special_tokens=False)
tokens = torch.tensor(token_ids, dtype=torch.long)
print(f"Total tokens available: {tokens.shape[0]:,}")

usable = min(TOKEN_BUDGET, tokens.shape[0])
n_seqs = usable // SEQ_LEN
usable = n_seqs * SEQ_LEN
assert usable > 0, "Not enough tokens."

tokens_2d = tokens[:usable].reshape(n_seqs, SEQ_LEN).to(device)
print(f"Using tokens: {usable:,}")
print(f"Token tensor: {tokens_2d.shape}")

## 5. Memory-safe helper functions

In [ ]:
WEIGHT_KEYWORDS = ["W_Q", "W_K", "W_V", "W_O", "W_in", "W_out"]


def restore_model(model, original_state):
    model.load_state_dict(original_state)
    model.eval()
    torch.cuda.empty_cache()
    gc.collect()


def is_quantizable(name):
    return any(s in name for s in WEIGHT_KEYWORDS)


def quantize_rtn_per_channel(model, bits):
    """
    Simulated per-channel RTN quantization.
    Quantizes selected TransformerLens weight matrices and dequantizes back to float dtype.
    """
    q_max = 2 ** (bits - 1) - 1
    q_min = -(2 ** (bits - 1))
    count = 0
    n_params = 0

    for name, param in model.named_parameters():
        if not is_quantizable(name):
            continue

        w = param.data
        # Match the previous smoke-test convention: reduce over all dims except the last.
        if w.ndim == 1:
            scale = w.abs().max() / q_max
        else:
            scale = w.abs().amax(dim=tuple(range(w.ndim - 1)), keepdim=True) / q_max
        scale = torch.clamp(scale, min=1e-12)

        q = torch.round(w / scale).clamp(q_min, q_max)
        param.data = (q * scale).to(w.dtype)
        count += 1
        n_params += w.numel()

    return count, n_params


def compute_perplexity(model, tokens_2d, batch_size):
    losses = []
    for i in tqdm(range(0, tokens_2d.shape[0], batch_size), desc="Perplexity", leave=False):
        batch = tokens_2d[i:i + batch_size]
        with torch.no_grad():
            loss = model(batch, return_type="loss")
        losses.append(float(loss.item()))
    avg_loss = float(np.mean(losses))
    return float(np.exp(avg_loss)), avg_loss


def get_hook_acts(model, batch, hook_name):
    """Return hook activations reshaped to (batch * seq_len, d_model)."""
    with torch.no_grad():
        _, cache = model.run_with_cache(batch, names_filter=[hook_name])
    acts = cache[hook_name].detach()
    acts = acts.reshape(-1, acts.shape[-1])
    del cache
    return acts


def encode_sae(sae, acts, device):
    with torch.no_grad():
        feats = sae.encode(acts.to(device).float())
    return feats


def pearson_from_sums(sum_x, sum_y, sum_x2, sum_y2, sum_xy, n):
    numerator = sum_xy - (sum_x * sum_y / n)
    denom_x = sum_x2 - (sum_x ** 2 / n)
    denom_y = sum_y2 - (sum_y ** 2 / n)
    denominator = torch.sqrt(torch.clamp(denom_x * denom_y, min=1e-12))
    corr = numerator / denominator
    return torch.clamp(corr, -1.0, 1.0)

print("Helpers ready.")

## 6. Streaming condition runner

In [ ]:
def run_condition_streaming(
    model,
    sae,
    tokens_2d,
    hook_name,
    device,
    original_state,
    bits=None,
    condition_name="FP16 baseline",
    batch_size=2,
    firing_threshold=0.001,
):
    """
    Memory-safe per-feature Pearson correlation.

    For each batch:
      1. Restore FP16, compute FP16 SAE features.
      2. Restore FP16 and optionally quantize, compute condition SAE features.
      3. Update running sums.
      4. Discard token-by-feature tensors.

    Returns summary dict and per-feature DataFrame. Does not return giant feature tensors.
    """
    d_sae = sae.cfg.d_sae

    # Running sums for Pearson correlation.
    sum_x = torch.zeros(d_sae, dtype=torch.float64)
    sum_y = torch.zeros(d_sae, dtype=torch.float64)
    sum_x2 = torch.zeros(d_sae, dtype=torch.float64)
    sum_y2 = torch.zeros(d_sae, dtype=torch.float64)
    sum_xy = torch.zeros(d_sae, dtype=torch.float64)

    # FP16 feature properties.
    fire_count = torch.zeros(d_sae, dtype=torch.float64)
    sum_x_activation = torch.zeros(d_sae, dtype=torch.float64)
    max_x_activation = torch.zeros(d_sae, dtype=torch.float64)

    total_positions = 0

    for i in tqdm(range(0, tokens_2d.shape[0], batch_size), desc=f"Streaming {condition_name}"):
        batch = tokens_2d[i:i + batch_size]

        # FP16 pass.
        restore_model(model, original_state)
        acts_fp16 = get_hook_acts(model, batch, hook_name)
        feats_fp16 = encode_sae(sae, acts_fp16, device).detach().cpu().to(torch.float64)

        # Quantized/current-condition pass.
        restore_model(model, original_state)
        if bits is not None and bits < 16:
            quantize_rtn_per_channel(model, bits=bits)
        acts_q = get_hook_acts(model, batch, hook_name)
        feats_q = encode_sae(sae, acts_q, device).detach().cpu().to(torch.float64)

        x = feats_fp16
        y = feats_q

        sum_x += x.sum(dim=0)
        sum_y += y.sum(dim=0)
        sum_x2 += (x ** 2).sum(dim=0)
        sum_y2 += (y ** 2).sum(dim=0)
        sum_xy += (x * y).sum(dim=0)

        fire_count += (x > 0).sum(dim=0)
        sum_x_activation += x.sum(dim=0)
        max_x_activation = torch.maximum(max_x_activation, x.max(dim=0).values)
        total_positions += x.shape[0]

        del acts_fp16, acts_q, feats_fp16, feats_q, x, y
        torch.cuda.empty_cache()
        gc.collect()

    restore_model(model, original_state)

    n = total_positions
    corr = pearson_from_sums(sum_x, sum_y, sum_x2, sum_y2, sum_xy, n)
    firing_rate = fire_count / n
    mean_activation = sum_x_activation / n
    active_mask = firing_rate > firing_threshold
    active_corrs = corr[active_mask]

    if active_corrs.numel() == 0:
        raise RuntimeError("No active features found. Lower FIRING_THRESHOLD.")

    summary = {
        "condition": condition_name,
        "bits": 16 if bits is None else bits,
        "n_tokens": int(n),
        "n_total_features": int(d_sae),
        "n_active_features": int(active_mask.sum().item()),
        "mean_corr": float(active_corrs.mean().item()),
        "median_corr": float(active_corrs.median().item()),
        "survived_>0.9_pct": float((active_corrs > 0.9).double().mean().item() * 100),
        "degraded_0.5_0.9_pct": float(((active_corrs > 0.5) & (active_corrs <= 0.9)).double().mean().item() * 100),
        "damaged_<0.5_pct": float((active_corrs < 0.5).double().mean().item() * 100),
    }

    per_feature = pd.DataFrame({
        "feature_id": np.arange(d_sae),
        "condition": condition_name,
        "bits": 16 if bits is None else bits,
        "corr": corr.numpy(),
        "firing_rate": firing_rate.numpy(),
        "mean_activation": mean_activation.numpy(),
        "max_activation": max_x_activation.numpy(),
        "active": active_mask.numpy(),
        "survived_>0.9": ((corr > 0.9) & active_mask).numpy(),
        "damaged_<0.5": ((corr < 0.5) & active_mask).numpy(),
    })

    return summary, per_feature

print("Streaming condition runner ready.")

## 7. Streaming layer sweep runner

In [ ]:
def run_sweep_for_layer_streaming(
    model,
    sae,
    hook_name,
    layer_idx,
    tokens_2d,
    original_state,
    bits_list,
    batch_size=2,
    firing_threshold=0.001,
    ckpt_prefix="phase2a",
):
    """Run FP16 + bitwidth sweep for one layer. Saves after every condition."""
    summary_path = CKPT(f"{ckpt_prefix}_summary.csv")
    per_feature_path = CKPT(f"{ckpt_prefix}_per_feature.csv")

    all_summaries = []
    all_per_feature = []

    # FP16 perplexity once.
    print(f"
=== Layer {layer_idx}: FP16 perplexity ===")
    restore_model(model, original_state)
    ppl_fp16, loss_fp16 = compute_perplexity(model, tokens_2d, batch_size=batch_size)
    print(f"FP16 ppl={ppl_fp16:.3f}, loss={loss_fp16:.4f}")

    # FP16 baseline feature pass. This will produce corr = 1 by comparing FP16 to FP16.
    print(f"
=== Layer {layer_idx}: FP16 feature baseline ===")
    summary_fp16, pf_fp16 = run_condition_streaming(
        model=model,
        sae=sae,
        tokens_2d=tokens_2d,
        hook_name=hook_name,
        device=device,
        original_state=original_state,
        bits=None,
        condition_name="FP16 baseline",
        batch_size=batch_size,
        firing_threshold=firing_threshold,
    )
    summary_fp16.update({
        "layer": layer_idx,
        "perplexity": ppl_fp16,
        "loss": loss_fp16,
        "ppl_delta_pct": 0.0,
    })
    pf_fp16["layer"] = layer_idx
    all_summaries.append(summary_fp16)
    all_per_feature.append(pf_fp16)

    pd.DataFrame(all_summaries).to_csv(summary_path, index=False)
    pd.concat(all_per_feature, ignore_index=True).to_csv(per_feature_path, index=False)

    # Quantized conditions.
    for bits in bits_list:
        condition_name = f"per-channel INT{bits}"
        print(f"
=== Layer {layer_idx}: {condition_name} ===")

        # Perplexity for quantized model.
        restore_model(model, original_state)
        n_tensors, n_params = quantize_rtn_per_channel(model, bits=bits)
        ppl_q, loss_q = compute_perplexity(model, tokens_2d, batch_size=batch_size)
        restore_model(model, original_state)

        print(f"Quantized {n_tensors} tensors / {n_params:,} params")
        print(f"{condition_name} ppl={ppl_q:.3f}, delta={(ppl_q / ppl_fp16 - 1) * 100:+.2f}%")

        # Feature correlations.
        summary_q, pf_q = run_condition_streaming(
            model=model,
            sae=sae,
            tokens_2d=tokens_2d,
            hook_name=hook_name,
            device=device,
            original_state=original_state,
            bits=bits,
            condition_name=condition_name,
            batch_size=batch_size,
            firing_threshold=firing_threshold,
        )
        summary_q.update({
            "layer": layer_idx,
            "perplexity": ppl_q,
            "loss": loss_q,
            "ppl_delta_pct": (ppl_q / ppl_fp16 - 1) * 100,
        })
        pf_q["layer"] = layer_idx

        all_summaries.append(summary_q)
        all_per_feature.append(pf_q)

        # Save after every condition so crashes do not lose completed results.
        pd.DataFrame(all_summaries).to_csv(summary_path, index=False)
        pd.concat(all_per_feature, ignore_index=True).to_csv(per_feature_path, index=False)

        print(pd.DataFrame(all_summaries).to_string(index=False))
        gc.collect()
        torch.cuda.empty_cache()

    results_df = pd.DataFrame(all_summaries)
    per_feature_df = pd.concat(all_per_feature, ignore_index=True)

    results_df.to_csv(summary_path, index=False)
    per_feature_df.to_csv(per_feature_path, index=False)
    restore_model(model, original_state)

    print(f"
Saved summary: {summary_path}")
    print(f"Saved per-feature metrics: {per_feature_path}")
    return results_df, per_feature_df

print("Streaming sweep runner ready.")

## 8. Phase 2A: primary layer

In [ ]:
hook_primary = f"blocks.{PRIMARY_LAYER}.hook_resid_post"
print("Loading primary SAE:", hook_primary)
sae_primary = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=hook_primary,
    device=device,
)
sae_primary.eval()
print(f"SAE loaded: d_in={sae_primary.cfg.d_in}, d_sae={sae_primary.cfg.d_sae}")

In [ ]:
results_2a, per_feature_2a = run_sweep_for_layer_streaming(
    model=model,
    sae=sae_primary,
    hook_name=hook_primary,
    layer_idx=PRIMARY_LAYER,
    tokens_2d=tokens_2d,
    original_state=original_state,
    bits_list=BITS_TO_TEST,
    batch_size=BATCH_SIZE,
    firing_threshold=FIRING_THRESHOLD,
    ckpt_prefix=f"phase2a_L{PRIMARY_LAYER}",
)

pd.set_option("display.float_format", "{:.3f}".format)
print("
=== Phase 2A summary ===")
print(results_2a.to_string(index=False))

## 9. Primary layer plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
quant = results_2a[results_2a["bits"] < 16].sort_values("bits", ascending=False)

axes[0].plot(quant["bits"], quant["ppl_delta_pct"], marker="o", markersize=10, linewidth=2, label="Perplexity Δ (%)")
axes[0].set_xlabel("Bits")
axes[0].set_ylabel("Perplexity Δ (%)")
axes[0].invert_xaxis()
axes[0].grid(True, alpha=0.3)
axes[0].set_title(f"Pythia-70M layer {PRIMARY_LAYER}: task degradation")

ax2 = axes[0].twinx()
ax2.plot(quant["bits"], quant["damaged_<0.5_pct"], marker="s", markersize=10, linewidth=2, linestyle="--", label="Damaged features (%)")
ax2.set_ylabel("Damaged features (%)")

axes[1].plot(quant["bits"], quant["survived_>0.9_pct"], marker="o", markersize=10, linewidth=2)
axes[1].set_xlabel("Bits")
axes[1].set_ylabel("Survived features (>0.9) (%)")
axes[1].invert_xaxis()
axes[1].grid(True, alpha=0.3)
axes[1].set_title("Feature survival vs bit-width")
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.savefig(CKPT(f"phase2a_L{PRIMARY_LAYER}_sweep.png"), dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(BITS_TO_TEST), figsize=(5 * len(BITS_TO_TEST), 5), sharey=True)
if len(BITS_TO_TEST) == 1:
    axes = [axes]

for ax, bits in zip(axes, BITS_TO_TEST):
    cond = f"per-channel INT{bits}"
    pf = per_feature_2a[(per_feature_2a["condition"] == cond) & (per_feature_2a["active"] == True)]
    row = results_2a[results_2a["condition"] == cond].iloc[0]

    ax.hist(pf["corr"].values, bins=60, edgecolor="black", alpha=0.8)
    ax.set_xlabel("Pearson correlation (vs FP16)")
    ax.set_title(f"{cond}
ppl Δ: {row['ppl_delta_pct']:+.2f}%, damaged: {row['damaged_<0.5_pct']:.1f}%")
    ax.axvline(0.9, linestyle="--", alpha=0.6)
    ax.axvline(0.5, linestyle="--", alpha=0.6)
    ax.set_xlim(0, 1.0)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Number of active features")
plt.tight_layout()
plt.savefig(CKPT(f"phase2a_L{PRIMARY_LAYER}_histograms.png"), dpi=140, bbox_inches="tight")
plt.show()

## 10. Lightweight disrupted-feature inspection

In [ ]:
# This does not use full token-by-feature activations. It only ranks damaged feature IDs.
damage_band = results_2a[(results_2a["bits"] < 16) & (results_2a["damaged_<0.5_pct"] > 0)]
if len(damage_band) > 0:
    target_cond = damage_band.iloc[0]["condition"]
else:
    target_cond = results_2a[results_2a["bits"] < 16].sort_values("survived_>0.9_pct").iloc[0]["condition"]

pf = per_feature_2a[(per_feature_2a["condition"] == target_cond) & (per_feature_2a["active"] == True)].sort_values("corr")

print(f"=== 20 most disrupted active features at {target_cond}, layer {PRIMARY_LAYER} ===")
print(
    pf[["feature_id", "corr", "firing_rate", "mean_activation", "max_activation", "survived_>0.9", "damaged_<0.5"]]
    .head(20)
    .to_string(index=False)
)

pf.head(50).to_csv(CKPT(f"phase2a_L{PRIMARY_LAYER}_top_disrupted_{target_cond.replace(' ', '_')}.csv"), index=False)

## 11. Phase 2A.5: secondary layer

In [ ]:
del sae_primary
try:
    del per_feature_2a
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print("Cleared primary SAE and large per-feature dataframe from memory.")

In [ ]:
hook_secondary = f"blocks.{SECONDARY_LAYER}.hook_resid_post"
print("Loading secondary SAE:", hook_secondary)
sae_secondary = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=hook_secondary,
    device=device,
)
sae_secondary.eval()
print(f"SAE loaded: d_in={sae_secondary.cfg.d_in}, d_sae={sae_secondary.cfg.d_sae}")

In [ ]:
results_2a5, per_feature_2a5 = run_sweep_for_layer_streaming(
    model=model,
    sae=sae_secondary,
    hook_name=hook_secondary,
    layer_idx=SECONDARY_LAYER,
    tokens_2d=tokens_2d,
    original_state=original_state,
    bits_list=BITS_TO_TEST,
    batch_size=BATCH_SIZE,
    firing_threshold=FIRING_THRESHOLD,
    ckpt_prefix=f"phase2a5_L{SECONDARY_LAYER}",
)

print("
=== Phase 2A.5 summary ===")
print(results_2a5.to_string(index=False))

## 12. Overlay layer comparison

In [ ]:
# Reload primary summary from disk because we deleted the per-feature dataframe.
results_2a_reload = pd.read_csv(CKPT(f"phase2a_L{PRIMARY_LAYER}_summary.csv"))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
q_primary = results_2a_reload[results_2a_reload["bits"] < 16].sort_values("bits", ascending=False)
q_secondary = results_2a5[results_2a5["bits"] < 16].sort_values("bits", ascending=False)

axes[0].plot(q_primary["bits"], q_primary["ppl_delta_pct"], marker="o", linewidth=2, label=f"L{PRIMARY_LAYER}")
axes[0].plot(q_secondary["bits"], q_secondary["ppl_delta_pct"], marker="s", linewidth=2, linestyle="--", label=f"L{SECONDARY_LAYER}")
axes[0].set_xlabel("Bits")
axes[0].set_ylabel("Perplexity Δ (%)")
axes[0].set_title("Task degradation")
axes[0].invert_xaxis()
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(q_primary["bits"], q_primary["survived_>0.9_pct"], marker="o", linewidth=2, label=f"L{PRIMARY_LAYER}")
axes[1].plot(q_secondary["bits"], q_secondary["survived_>0.9_pct"], marker="s", linewidth=2, linestyle="--", label=f"L{SECONDARY_LAYER}")
axes[1].set_xlabel("Bits")
axes[1].set_ylabel("Survived (>0.9) (%)")
axes[1].set_title("Feature survival by layer")
axes[1].invert_xaxis()
axes[1].grid(True, alpha=0.3)
axes[1].legend()
axes[1].set_ylim(0, 105)

axes[2].plot(q_primary["bits"], q_primary["damaged_<0.5_pct"], marker="o", linewidth=2, label=f"L{PRIMARY_LAYER}")
axes[2].plot(q_secondary["bits"], q_secondary["damaged_<0.5_pct"], marker="s", linewidth=2, linestyle="--", label=f"L{SECONDARY_LAYER}")
axes[2].set_xlabel("Bits")
axes[2].set_ylabel("Damaged (<0.5) (%)")
axes[2].set_title("Feature damage by layer")
axes[2].invert_xaxis()
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.savefig(CKPT("phase2a_layer_comparison.png"), dpi=140, bbox_inches="tight")
plt.show()

## 13. Combined results table

In [ ]:
results_2a_reload = pd.read_csv(CKPT(f"phase2a_L{PRIMARY_LAYER}_summary.csv"))
results_2a_reload["layer"] = PRIMARY_LAYER
results_2a5["layer"] = SECONDARY_LAYER

combined = pd.concat([results_2a_reload, results_2a5], ignore_index=True)
combined = combined[[
    "layer", "condition", "bits", "perplexity", "ppl_delta_pct",
    "n_active_features", "mean_corr", "median_corr",
    "survived_>0.9_pct", "degraded_0.5_0.9_pct", "damaged_<0.5_pct",
]]
combined.to_csv(CKPT("phase2a_combined.csv"), index=False)

pd.set_option("display.max_columns", None)
print(combined.to_string(index=False))
print(f"
Saved: {CKPT('phase2a_combined.csv')}")

## Notes

- This notebook intentionally does **not** save `features_fp16.pt` or full feature tensors.
- Phase 5 top-context inspection should rerun only the top-k damaged feature IDs on a much smaller subset, not the full 200k/500k tensor.
- If the kernel still dies, reduce `BATCH_SIZE` to `1` and/or reduce `TOKEN_BUDGET` temporarily to debug.